# Extra after Lab 3: Clinical evaluation of AI tools

**AI2Practice Research Lab** | about 45-60 minutes | optional

**The question.** In Lab 3 you judged the pneumothorax detector with sensitivity, specificity and an ROC curve. But which numbers *should* a validation report contain? Picking metrics by habit can hide real problems. This extra notebook walks through **Metrics Reloaded**, a published framework that helps researchers choose validation metrics for image analysis tools, and applies it to the same detector.

By the end you will be able to describe a validation problem as a "problem fingerprint", explain why accuracy can mislead, and compute balanced accuracy, the Matthews correlation coefficient, Expected Cost, AUROC, Average Precision, the Brier score and Net Benefit. You need Lab 3 (and your `sens_spec()` from it).

This notebook has 5 parts: Part 1, the problem fingerprint; Part 2, metrics at one threshold; Part 3, metrics across all thresholds; Part 4, calibration and Net Benefit; Part 5, comparing the two sites.

**The scenario.** Same detector, same two sites as Lab 3: A (the main hospital) and B (a smaller affiliated hospital). The department now wants a validation report it can defend. Two extra facts for this exercise: a missed pneumothorax is judged much worse than a false alarm, and the department would like to show radiologists the AI score as if it were a probability ("62% chance of pneumothorax"). Both facts matter for which metrics you choose.

#### How the tasks work

Each task is tagged with a **mode** that tells you what kind of work it is:

| Mode | What you do |
|---|---|
| **Write** | Write a short piece of code in an empty cell. Try it on your own first; a hint and a hidden solution are there if you need them. |
| **Fill in** | Some code is written for you. Replace each `____` with the missing piece. |
| **Reuse** | Import a function you wrote in an earlier lab from your own `toolkit.py`. |
| **Direct the AI** | Ask an AI assistant to write the code, paste it in, run the test, and fill in a short AI-use log. |
| **Fix** | Code "written by an AI" has one bug. Find it and fix it. |
| **Read** | Read some code and predict what it does before you run it. |

Every task ends with a **test cell**. Run it after your code. If it prints `Check passed`, move on. If it shows an `AssertionError`, the message under it tells you what to look at; change your code and run both cells again.

## Set your GitHub username

Your practice data is generated from your GitHub username, so everyone gets a slightly different dataset and the lab page can check your own numbers. Type your username between the quotes (the part after `github.com/` in your profile address), then run the cell with **Shift + Enter**.

In [ ]:
GITHUB_USERNAME = "your-username"   # <- change this to your GitHub username

## Setup

The next cell gets everything ready and loads the same detector data you used in Lab 3. It also imports some metric functions from **scikit-learn**, the standard Python machine learning library. Run it once each time you open the notebook. It downloads two small helper files from the course repository if they are not here yet: `rl_data.py` makes your practice data, and `toolkit.py` is your own function library (more on that later). Lines starting with `!` are commands for the computer Colab runs on, not Python.

In [ ]:
# Setup: get the helper files (only if they are missing), then make your data
import os
COURSE = "https://raw.githubusercontent.com/UM-IHC-CA2i/CA2i-Research-Launchpad/main/lesson-notebooks"
FORK = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/CA2i-Research-Launchpad/main/lesson-notebooks"
if not os.path.exists("rl_data.py"):
    !wget -q {COURSE}/rl_data.py
if not os.path.exists("toolkit.py"):
    # Your fork first, so functions you saved in earlier labs come back; else the course copy.
    !wget -q -O toolkit.py {FORK}/toolkit.py || wget -q -O toolkit.py {COURSE}/toolkit.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rl_data
import toolkit
from toolkit import save_to_toolkit

if GITHUB_USERNAME.strip().lower() in ("", "your-username"):
    print("Reminder: put your own GitHub username in the cell above, then run both cells again.")
det = rl_data.dataset(3, GITHUB_USERNAME)   # same detector data as Lab 3
from sklearn.metrics import (roc_auc_score, average_precision_score, matthews_corrcoef,
                             balanced_accuracy_score, brier_score_loss)
from sklearn.calibration import calibration_curve
print("Ready. Data made for GitHub user:", rl_data.normalize_username(GITHUB_USERNAME))

---
## Part 1: The problem fingerprint

**Metrics Reloaded** (Maier-Hein, Reinke and colleagues, *Nature Methods* 2024) came out of a large international expert consensus. Its starting point: the right metric depends on the problem, so describe the problem first. It sorts image analysis problems into four **categories**:

- **Image-level classification**: one answer per image (pneumothorax yes or no).
- **Object detection**: find and locate each object, for example a box around each lung nodule.
- **Semantic segmentation**: label every pixel, for example "lung" or "not lung".
- **Instance segmentation**: label every pixel *and* tell separate objects apart, for example each nodule outlined on its own.

The framework then works in three steps:

1. **Problem fingerprint.** A structured description of the problem: the domain interest (what matters clinically), the properties of the target structure, properties of the dataset such as **prevalence** (the share of cases with the condition) and **class imbalance** (one class much more common than the other), what the algorithm outputs, whether a **decision threshold** exists (a cut-off that turns a score into yes or no), and whether **calibration** matters (whether the scores should be read as real probabilities).
2. **Metric selection.** The fingerprint guides you to suitable metrics.
3. **Application.** Compute the metrics and report them properly.

You can try the guided version yourself in the online tool at https://metrics-reloaded.dkfz.de/ (it opens in your browser; nothing to install).

In [ ]:
print("Cases:", len(det), " with pneumothorax:", det["truth"].sum())
det.head()

**Mode: Fill in**

#### Task 1. Fill in the detector's fingerprint

Complete the fingerprint for the pneumothorax detector. Replace each `____`. For the text answers, pick from the options in the comments. Compute the test-set prevalence from `det` (the mean of a 0/1 column is the share of 1s).

In [ ]:
fingerprint = {
    "category": "____",              # "image-level classification", "object detection", "semantic segmentation", "instance segmentation"
    "prevalence_test": ____,         # share of cases with pneumothorax in det
    "class_imbalance": ____,         # True or False: is one class clearly more common?
    "output": "____",                # "score" (a number from 0 to 1) or "label" (just 0 or 1)
    "decision_threshold": ____,      # True or False: will a cut-off turn scores into yes/no?
    "unequal_costs": ____,           # True or False: are some mistakes worse than others?
    "calibration_matters": ____,     # True or False: should scores be read as probabilities?
}
fingerprint

<details><summary>Show the answer</summary>

One answer per X-ray makes this image-level classification. About a third of the cases are positive, so the classes are imbalanced. The tool outputs a score, the department will use a cut-off, a miss is worse than a false alarm, and showing scores as probabilities means calibration matters.

```python
fingerprint = {
    "category": "image-level classification",
    "prevalence_test": det["truth"].mean(),
    "class_imbalance": True,
    "output": "score",
    "decision_threshold": True,
    "unequal_costs": True,
    "calibration_matters": True,
}
fingerprint
```

</details>

In [ ]:
assert fingerprint["category"].strip().lower() == "image-level classification", "One yes/no answer per X-ray: which category is that?"
assert abs(fingerprint["prevalence_test"] - det["truth"].mean()) < 1e-9, 'prevalence_test should be det["truth"].mean().'
assert fingerprint["class_imbalance"] is True, "About a third of cases are positive, so negatives clearly outnumber positives."
assert fingerprint["output"].strip().lower() == "score", "The tool gives a score from 0 to 1."
assert fingerprint["decision_threshold"] is True, "The department will flag cases above a cut-off (0.5 in Lab 3)."
assert fingerprint["unequal_costs"] is True, "The scenario says a miss is much worse than a false alarm."
assert fingerprint["calibration_matters"] is True, "The department wants to show scores as probabilities."
print("Check passed")

### Step 2 for image-level classification

For this category, Metrics Reloaded recommends, in short:

- **Per-threshold metrics** (computed after the cut-off is applied). **Expected Cost** is generally recommended when different mistakes carry different costs. Otherwise **Accuracy**, **Balanced Accuracy** or the **Matthews correlation coefficient (MCC)**, depending on whether the prevalence in the test set matches the prevalence in the target population (the patients the tool will really be used on). Per-class metrics such as the **F-beta score** can be added if wanted.
- **Multi-threshold metrics** (computed from the scores across all possible cut-offs) are always recommended. **AUROC** (area under the ROC curve, the AUC from Lab 3) is the default because it does not depend on prevalence. **Average Precision** is an alternative in some class-balance settings.
- **Calibration metrics**, such as the **Brier score**, if calibration is requested.
- **Net Benefit** is highlighted when the question is a cost-benefit decision: is using the tool better than treating everyone or no one?

The rest of this notebook computes each of these for the detector.

---
## Part 2: Metrics at one threshold

**Accuracy** is the share of all cases the tool gets right: (TP + TN) / all cases. It sounds like the obvious metric, but it hides a trap when one class is rare.

Our test set is **enriched**: cases were collected so that about a third have a pneumothorax. In the hospital's everyday stream of chest X-rays a pneumothorax is far rarer. For this exercise, assume 20 in every 1,000 X-rays (2%).

**Mode: Write**

#### Task 2. The model that never says yes

Imagine a useless "model" that calls every X-ray negative. Below, `truth_low` holds 1,000 cases with 20 positives, and `pred_never` is all zeros. Compute its accuracy as a fraction (correct answers divided by all cases) and store it in `acc_never`.

*Try this on your own first, without an AI assistant (you can switch off Colab's automatic code suggestions under Tools > Settings > AI assistance). Use the hint if you get stuck, and open the solution only after a real attempt.*

*Hint:* Walk through both lists together with `zip(truth_low, pred_never)` and count the pairs where the two are equal. Then divide by `len(truth_low)`.

In [ ]:
truth_low = [1] * 20 + [0] * 980
pred_never = [0] * 1000

# Your code here


print("Accuracy of the always-negative model:", acc_never)

<details><summary>Show a solution</summary>

```python
truth_low = [1] * 20 + [0] * 980
pred_never = [0] * 1000

correct = sum(1 for t, p in zip(truth_low, pred_never) if t == p)
acc_never = correct / len(truth_low)

print("Accuracy of the always-negative model:", acc_never)
```

</details>

In [ ]:
assert abs(acc_never - 0.98) < 1e-9, "Count the cases where pred_never equals truth_low, then divide by 1000."
print("Check passed")

98% accuracy, and it never finds a single pneumothorax. Its sensitivity is 0. This is why accuracy alone is a poor choice when classes are imbalanced, and why the framework asks about prevalence in the fingerprint.

Two alternatives:

- **Balanced accuracy** = (sensitivity + specificity) / 2. It weighs the two classes equally, so it does not change with prevalence. The always-negative model gets (0 + 1) / 2 = 0.5, no better than a coin toss.
- **Matthews correlation coefficient (MCC)** uses all four boxes of the confusion matrix. It runs from -1 (always wrong) through 0 (no better than chance) to +1 (perfect).

**Mode: Reuse**

#### Task 3. Confusion matrix with sens_spec()

Turn the scores into yes/no answers at the Lab 3 threshold of 0.5 and run your `sens_spec()` from Lab 3 on them. If the import fails with "still a stub", your saved `toolkit.py` did not reach your fork. Open your Lab 3 notebook, copy your `sens_spec()` cell here, run it, and carry on (then save it with `save_to_toolkit(sens_spec)` and upload `toolkit.py` as in the Save to GitHub steps).

In [ ]:
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import sens_spec

det["pred"] = (det["score"] >= 0.5).astype(int)
m = ____
m

<details><summary>Show a solution</summary>

```python
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import sens_spec

det["pred"] = (det["score"] >= 0.5).astype(int)
m = sens_spec(det["truth"], det["pred"])
m
```

</details>

In [ ]:
assert rl_data.check(3, GITHUB_USERNAME, "tp", m["tp"]), "Use sens_spec(det['truth'], det['pred'])."
assert rl_data.check(3, GITHUB_USERNAME, "sensitivity", m["sensitivity"])
print("Check passed")

**Mode: Fill in**

#### Task 4. Balanced accuracy and MCC

Compute balanced accuracy from `m`, and MCC from the four counts. The MCC formula is long, so most of it is written for you. Afterwards the cell compares your numbers with scikit-learn's.

In [ ]:
import math
tp, fp, fn, tn = m["tp"], m["fp"], m["fn"], m["tn"]

bal_acc = (m["sensitivity"] + ____) / 2
mcc = (tp * tn - fp * fn) / math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (____))

print("Balanced accuracy:", round(bal_acc, 3), " scikit-learn:", round(balanced_accuracy_score(det["truth"], det["pred"]), 3))
print("MCC:", round(mcc, 3), " scikit-learn:", round(matthews_corrcoef(det["truth"], det["pred"]), 3))

<details><summary>Show a solution</summary>

```python
import math
tp, fp, fn, tn = m["tp"], m["fp"], m["fn"], m["tn"]

bal_acc = (m["sensitivity"] + m["specificity"]) / 2
mcc = (tp * tn - fp * fn) / math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))

print("Balanced accuracy:", round(bal_acc, 3), " scikit-learn:", round(balanced_accuracy_score(det["truth"], det["pred"]), 3))
print("MCC:", round(mcc, 3), " scikit-learn:", round(matthews_corrcoef(det["truth"], det["pred"]), 3))
```

</details>

In [ ]:
assert abs(bal_acc - balanced_accuracy_score(det["truth"], det["pred"])) < 1e-9, "Balanced accuracy is the mean of sensitivity and specificity."
assert abs(mcc - matthews_corrcoef(det["truth"], det["pred"])) < 1e-9, "The last bracket of the MCC denominator is (tn + fn)."
print("Check passed")

### Expected Cost

The scenario says a missed pneumothorax is much worse than a false alarm. Balanced accuracy and MCC treat both mistakes the same. **Expected Cost** lets you say how bad each mistake is, using a **cost matrix**. For this exercise the department agreed on:

| | Truth: pneumothorax | Truth: no pneumothorax |
|---|---|---|
| **AI says positive** | 0 (correct) | 1 (false alarm) |
| **AI says negative** | 5 (missed pneumothorax) | 0 (correct) |

In its simplest form, Expected Cost is the average cost per case: (5 x FN + 1 x FP) / number of cases. Lower is better, and 0 would be perfect. The numbers 5 and 1 are a choice the clinical team has to make and justify; they are not a fact about the tool.

**Mode: Direct the AI**

#### Task 5. Expected Cost

Ask an AI assistant to write a function `expected_cost(truth, pred, cost_fn=5, cost_fp=1)` and use it on the detector. Also compute the Expected Cost of two simple strategies for comparison: calling everything negative and calling everything positive. Paste the code below, run it, then run the test. Read the code before you trust it.

Suggested prompt for your AI assistant (adapt it as you like):

> I have a pandas DataFrame `det` with columns `truth` (0 or 1) and `pred` (0 or 1).
> Write a Python function expected_cost(truth, pred, cost_fn=5, cost_fp=1) that returns
> (cost_fn * number of false negatives + cost_fp * number of false positives) / number of cases.
> It should work for lists and pandas columns. Then compute:
> ec_tool = expected_cost(det["truth"], det["pred"]),
> ec_all_neg for predicting 0 for every case, and ec_all_pos for predicting 1 for every case. Print all three.

In [ ]:
# Paste the AI's code here

<details><summary>Show a reference solution (only after trying with the AI)</summary>

```python
def expected_cost(truth, pred, cost_fn=5, cost_fp=1):
    """Average cost per case for 0/1 predictions."""
    truth, pred = list(truth), list(pred)
    fn = sum(1 for t, p in zip(truth, pred) if t == 1 and p == 0)
    fp = sum(1 for t, p in zip(truth, pred) if t == 0 and p == 1)
    return (cost_fn * fn + cost_fp * fp) / len(truth)

ec_tool = expected_cost(det["truth"], det["pred"])
ec_all_neg = expected_cost(det["truth"], [0] * len(det))
ec_all_pos = expected_cost(det["truth"], [1] * len(det))
print("Expected Cost  tool:", round(ec_tool, 3), " all negative:", round(ec_all_neg, 3), " all positive:", round(ec_all_pos, 3))
```

</details>

In [ ]:
t = det["truth"]
assert abs(ec_tool - (5 * ((det["pred"] == 0) & (t == 1)).sum() + ((det["pred"] == 1) & (t == 0)).sum()) / len(det)) < 1e-9, "ec_tool should be (5*FN + 1*FP) / number of cases."
assert abs(ec_all_neg - 5 * t.mean()) < 1e-9, "Calling everything negative misses every positive: cost 5 for each one."
assert abs(ec_all_pos - (1 - t.mean())) < 1e-9, "Calling everything positive gives a false alarm for every negative: cost 1 each."
assert expected_cost([1, 0], [0, 1], cost_fn=2, cost_fp=3) == 2.5, "expected_cost() should use the cost_fn and cost_fp arguments."
print("Check passed")

**AI-use log (Task 5).** Double-click this cell to edit it.

- Tool I used:
- My prompt (paste it):
- What I changed in the AI's code, and why:
- Did the test cell pass on the first try? If not, what was wrong?

---
## Part 3: Metrics across all thresholds

Per-threshold metrics depend on the cut-off you picked. **Multi-threshold metrics** use the raw scores and look at every possible cut-off at once, which is why the framework always recommends one.

- **AUROC** (area under the ROC curve) is the AUC from Lab 3. It is the default because it does not depend on prevalence. 0.5 is a coin toss, 1.0 is perfect.
- **Average Precision (AP)** summarizes the **precision-recall curve**. Precision is another name for PPV, and recall is another name for sensitivity. AP does depend on prevalence: a tool that guesses at random gets an AP about equal to the prevalence, not 0.5. So an AP of 0.6 means something different in a test set with 30% prevalence than in one with 2%.

**Mode: Fill in**

#### Task 6. AUROC and Average Precision

Both scikit-learn functions take the truth first and the **score** second. Fill in the blanks.

In [ ]:
auroc = roc_auc_score(det["truth"], ____)
ap = average_precision_score(det["truth"], ____)
print(f"AUROC = {auroc:.3f}")
print(f"Average Precision = {ap:.3f}  (a random guess would get about {det['truth'].mean():.3f})")

<details><summary>Show a solution</summary>

```python
auroc = roc_auc_score(det["truth"], det["score"])
ap = average_precision_score(det["truth"], det["score"])
print(f"AUROC = {auroc:.3f}")
print(f"Average Precision = {ap:.3f}  (a random guess would get about {det['truth'].mean():.3f})")
```

</details>

In [ ]:
assert abs(auroc - roc_auc_score(det["truth"], det["score"])) < 1e-12, 'Use det["score"], not the 0/1 prediction.'
assert abs(ap - average_precision_score(det["truth"], det["score"])) < 1e-12, 'Use det["score"] for Average Precision too.'
print("Check passed")

**Mode: Fix**

#### Task 7. An AUROC that is too low

A colleague asked an AI assistant for the AUROC at each site. The code runs, but the numbers come out lower than they should. Find the one bug and fix it in place.

In [ ]:
# Written by an AI assistant
auroc_by_site = {}
for site in ["A", "B"]:
    rows = det[det["site"] == site]
    auroc_by_site[site] = roc_auc_score(rows["truth"], rows["pred"])
auroc_by_site

<details><summary>Show the bug and the fix</summary>

The code passes `pred` (the 0/1 answers after the 0.5 cut-off) instead of `score`. With only two values there is only one threshold left, so the "curve" collapses to a single point joined to the corners, and the area shrinks. AUROC must be computed from the scores. This mistake is easy to make and easy to miss, because the code still returns a plausible number.

```python
# Written by an AI assistant (fixed)
auroc_by_site = {}
for site in ["A", "B"]:
    rows = det[det["site"] == site]
    auroc_by_site[site] = roc_auc_score(rows["truth"], rows["score"])
auroc_by_site
```

</details>

In [ ]:
for site in ["A", "B"]:
    rows = det[det["site"] == site]
    assert abs(auroc_by_site[site] - roc_auc_score(rows["truth"], rows["score"])) < 1e-12, f"AUROC for site {site} is off. What does the second argument of roc_auc_score() need to be?"
print("Check passed")

---
## Part 4: Calibration and Net Benefit

A tool is **well calibrated** if its scores can be read as probabilities: among all cases scored around 0.3, about 30% really have a pneumothorax. AUROC cannot tell you this, because it only looks at the ranking of the scores. The department wants to show scores as probabilities, so the fingerprint says calibration matters.

The **Brier score** is the average of (score - truth) squared over all cases. 0 is perfect; lower is better. A **calibration plot** groups cases by score and plots the average score in each group against the share that really had a pneumothorax. A well-calibrated tool lies on the diagonal.

**Mode: Fill in**

#### Task 8. Brier score and a calibration plot

Compute the Brier score yourself, then fill in the call to `calibration_curve()`, which groups the cases into 10 bins of score. It returns the observed share of positives and the mean score in each bin.

In [ ]:
brier = ((det["score"] - det["truth"]) ** 2).____()
print("Brier score:", round(brier, 4), " scikit-learn:", round(brier_score_loss(det["truth"], det["score"]), 4))

observed, predicted = calibration_curve(det["truth"], det[____], n_bins=10)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(predicted, observed, "o-", color="#0D9488", label="AI tool")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.set_xlabel("Mean AI score in the bin")
ax.set_ylabel("Share with pneumothorax in the bin")
ax.set_title("Calibration plot, pneumothorax detector")
ax.legend()
plt.show()

<details><summary>Show a solution</summary>

```python
brier = ((det["score"] - det["truth"]) ** 2).mean()
print("Brier score:", round(brier, 4), " scikit-learn:", round(brier_score_loss(det["truth"], det["score"]), 4))

observed, predicted = calibration_curve(det["truth"], det["score"], n_bins=10)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(predicted, observed, "o-", color="#0D9488", label="AI tool")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.set_xlabel("Mean AI score in the bin")
ax.set_ylabel("Share with pneumothorax in the bin")
ax.set_title("Calibration plot, pneumothorax detector")
ax.legend()
plt.show()
```

</details>

In [ ]:
assert abs(brier - brier_score_loss(det["truth"], det["score"])) < 1e-12, "The Brier score is the mean of the squared differences: .mean()."
assert len(observed) == len(predicted) and len(observed) >= 5, 'Pass det["score"] to calibration_curve().'
print("Check passed")

Look at the plot. Where the line runs above the diagonal, the tool's scores are too low (more cases are positive than the score suggests); below the diagonal, they are too high. Would you be comfortable showing these scores to radiologists as "percent chance"?

### Net Benefit

**Net Benefit** asks a decision question: if we act on the tool's flags (for example, a radiologist reviews the case urgently), is that better than acting on everyone or on no one? It starts from a **threshold probability** p: the risk at which you would act. Choosing p also says how you weigh the two mistakes: at p = 0.2, one true positive is worth the same as 0.2 / 0.8 = 0.25 false positives' cost, so you accept 4 false alarms per pneumothorax found.

Net Benefit = TP / N - (FP / N) x p / (1 - p), where N is the number of cases and the tool flags a case when score >= p.

For comparison, "act on everyone" has Net Benefit = prevalence - (1 - prevalence) x p / (1 - p), and "act on no one" has Net Benefit 0. A tool is only useful at that p if it beats both. Strictly, Net Benefit treats the score as a probability, so it is most meaningful for a well-calibrated tool, which ties back to the plot above.

**Mode: Fill in**

#### Task 9. Net Benefit at two threshold probabilities

Complete `net_benefit()` and run it at p = 0.1 and p = 0.2, next to the "act on everyone" strategy.

In [ ]:
def net_benefit(truth, score, p):
    """Net Benefit of acting when score >= p."""
    truth = np.asarray(truth)
    flag = np.asarray(score) >= p
    n = len(truth)
    tp = (flag & (truth == 1)).sum()
    fp = (flag & (truth == 0)).sum()
    return tp / n - (fp / n) * ____


def net_benefit_all(truth, p):
    prev = np.mean(truth)
    return prev - (1 - prev) * p / (1 - p)


nb_results = {}
for p in [0.1, 0.2]:
    nb_results[p] = {"tool": net_benefit(det["truth"], det["score"], p),
                     "act on everyone": net_benefit_all(det["truth"], p),
                     "act on no one": 0.0}
pd.DataFrame(nb_results).T.round(3)

<details><summary>Show a solution</summary>

```python
def net_benefit(truth, score, p):
    """Net Benefit of acting when score >= p."""
    truth = np.asarray(truth)
    flag = np.asarray(score) >= p
    n = len(truth)
    tp = (flag & (truth == 1)).sum()
    fp = (flag & (truth == 0)).sum()
    return tp / n - (fp / n) * p / (1 - p)


def net_benefit_all(truth, p):
    prev = np.mean(truth)
    return prev - (1 - prev) * p / (1 - p)


nb_results = {}
for p in [0.1, 0.2]:
    nb_results[p] = {"tool": net_benefit(det["truth"], det["score"], p),
                     "act on everyone": net_benefit_all(det["truth"], p),
                     "act on no one": 0.0}
pd.DataFrame(nb_results).T.round(3)
```

</details>

In [ ]:
assert abs(net_benefit([1, 0, 1, 0], [0.9, 0.8, 0.1, 0.1], 0.2) - (1 / 4 - (1 / 4) * 0.25)) < 1e-9, "The weight for false positives is p / (1 - p)."
for p in [0.1, 0.2]:
    flag = det["score"] >= p
    want = (flag & (det["truth"] == 1)).sum() / len(det) - (flag & (det["truth"] == 0)).sum() / len(det) * p / (1 - p)
    assert abs(nb_results[p]["tool"] - want) < 1e-9
print("Check passed")

---
## Part 5: Comparing the two sites

In Lab 3 the detector was less sensitive at site B. Do the metrics recommended by the framework tell the same story? Each one looks at a different side of performance, so it is worth putting them side by side.

**Mode: Fill in**

#### Task 10. One table for both sites

Complete `site_metrics()`, which computes the main metrics from this notebook for one group of cases, and build a table with one row per site.

In [ ]:
def site_metrics(rows):
    pred = (rows["score"] >= 0.5).astype(int)
    r = sens_spec(rows["truth"], pred)
    return {
        "cases": len(rows),
        "prevalence": rows["truth"].mean(),
        "sensitivity": r["sensitivity"],
        "balanced accuracy": (r["sensitivity"] + r["specificity"]) / 2,
        "MCC": matthews_corrcoef(rows["truth"], pred),
        "Expected Cost": expected_cost(rows["truth"], pred),
        "AUROC": roc_auc_score(rows["truth"], ____),
        "Average Precision": average_precision_score(rows["truth"], rows["score"]),
        "Brier": brier_score_loss(rows["truth"], rows["score"]),
        "Net Benefit (p=0.2)": net_benefit(rows["truth"], rows["score"], 0.2),
    }

by_site = pd.DataFrame({site: site_metrics(det[det["site"] == ____]) for site in ["A", "B"]}).T
by_site.round(3)

<details><summary>Show a solution</summary>

```python
def site_metrics(rows):
    pred = (rows["score"] >= 0.5).astype(int)
    r = sens_spec(rows["truth"], pred)
    return {
        "cases": len(rows),
        "prevalence": rows["truth"].mean(),
        "sensitivity": r["sensitivity"],
        "balanced accuracy": (r["sensitivity"] + r["specificity"]) / 2,
        "MCC": matthews_corrcoef(rows["truth"], pred),
        "Expected Cost": expected_cost(rows["truth"], pred),
        "AUROC": roc_auc_score(rows["truth"], rows["score"]),
        "Average Precision": average_precision_score(rows["truth"], rows["score"]),
        "Brier": brier_score_loss(rows["truth"], rows["score"]),
        "Net Benefit (p=0.2)": net_benefit(rows["truth"], rows["score"], 0.2),
    }

by_site = pd.DataFrame({site: site_metrics(det[det["site"] == site]) for site in ["A", "B"]}).T
by_site.round(3)
```

</details>

In [ ]:
assert list(by_site.index) == ["A", "B"]
b = det[det["site"] == "B"]
assert abs(by_site.loc["B", "AUROC"] - roc_auc_score(b["truth"], b["score"])) < 1e-9, 'AUROC needs rows["score"], and each row should use only that site\'s cases.'
assert rl_data.check(3, GITHUB_USERNAME, "sens_b", by_site.loc["B", "sensitivity"])
print("Check passed")

Read the table row by row. Is site B worse on every metric, or only on some? Remember which metrics depend on prevalence (accuracy, PPV, Average Precision, Expected Cost in this simple form, Net Benefit) and which do not (sensitivity, specificity, balanced accuracy, AUROC). If the two sites have different prevalence, a prevalence-dependent metric can differ even when the tool itself behaves the same. With fewer cases at site B, its numbers are also less certain; a full report would add **confidence intervals** (a range of plausible values), which you can ask your AI assistant to compute by bootstrapping (resampling the cases many times).

---
## Optional: the MetricsReloaded Python package

Everything above used scikit-learn only. The framework's authors also maintain an open-source Python package, **MetricsReloaded** (https://github.com/Project-MONAI/MetricsReloaded), which implements the metrics, including ones for detection and segmentation. If you are curious, run the cell below to install it, then look at the package's README on GitHub for how to call its functions. This step is optional and not needed for anything else in this notebook; installing takes a minute or two.

In [ ]:
# OPTIONAL: install the MetricsReloaded package (not needed for this notebook)
!pip install -q git+https://github.com/Project-MONAI/MetricsReloaded

---
## What to report

Following the three steps, a validation report for this detector would include:

1. **The problem fingerprint**: image-level classification; test-set prevalence (and how it differs from the target population); class imbalance; score output with a decision threshold; unequal costs of mistakes; calibration requested.
2. **Per-threshold metrics** at the stated threshold: Expected Cost with the cost matrix written out and justified, plus balanced accuracy or MCC (not accuracy alone), and sensitivity and specificity for clinical readers.
3. **A multi-threshold metric**: AUROC, and Average Precision with the prevalence stated next to it.
4. **Calibration**: the Brier score and a calibration plot, because scores will be shown as probabilities.
5. **Net Benefit** at one or two threshold probabilities the clinical team agrees on, compared with acting on everyone and on no one.
6. **Subgroups**: the same metrics for each site, with the number of cases in each.

A sentence for a report might read (with your own numbers): "At a threshold of 0.5 the Expected Cost was 0.28 (cost 5 per missed pneumothorax, 1 per false alarm), against 1.55 for calling all cases negative; AUROC was 0.96 overall but lower at site B, and the calibration plot showed that scores did not match observed risk (cases scored around 0.25 were almost never positive)."

The companion paper on pitfalls (Reinke and colleagues, 2024) is worth a read before you write your own validation section: it collects common ways metrics mislead.

---
## Save to GitHub

In Colab, choose **File > Save a copy in GitHub**. The first time, Colab asks you to sign in to GitHub and allow access. Then set:

- Repository: `<your-username>/CA2i-Research-Launchpad` (your fork)
- Branch: `main`
- File path: `lesson-notebooks/Extra_Clinical_Evaluation.ipynb`
- Commit message: `Extra: clinical evaluation finished`

Leave "Include a link to Colab" ticked and click **OK**. A **commit** is a saved snapshot of your files with a short message, so you can always see what changed and go back.

---
## Sources

- Maier-Hein L, Reinke A, Godau P, et al. Metrics reloaded: recommendations for image analysis validation. *Nat Methods*. 2024;21(2):195-212. doi:10.1038/s41592-023-02151-z. PubMed 38347141: https://pubmed.ncbi.nlm.nih.gov/38347141/
- Reinke A, Tizabi MD, Baumgartner M, et al. Understanding metric-related pitfalls in image analysis validation. *Nat Methods*. 2024;21(2):182-194. doi:10.1038/s41592-023-02150-0
- Metrics Reloaded online tool: https://metrics-reloaded.dkfz.de/
- MetricsReloaded Python package: https://github.com/Project-MONAI/MetricsReloaded